In [27]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer
import numpy as np

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\adiko\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\adiko\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [28]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
631,I agree with what so many others have said abo...,negative
919,This movie lacks in everything. Except Bobby d...,negative
591,"The best thing about the movie is the name, as...",negative
96,Gene Hackman gets himself busted out of prison...,negative
296,When I first watched this show on Cartoon Netw...,negative


In [29]:

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [30]:
df = normalize_text(df)
df.head()

,review,sentiment
631,agree many others said shallow offensive natur...,negative
919,movie lack everything except bobby deol standa...,negative
591,best thing movie name describes plot acting le...,negative
96,gene hackman get busted prison nameless govern...,negative
296,first watched show cartoon network found unint...,negative


In [31]:
df['sentiment'].value_counts()

sentiment
positive    251
negative    249
Name: count, dtype: int64

In [32]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [33]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
631,agree many others said shallow offensive natur...,0
919,movie lack everything except bobby deol standa...,0
591,best thing movie name describes plot acting le...,0
96,gene hackman get busted prison nameless govern...,0
296,first watched show cartoon network found unint...,0


In [34]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [35]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [36]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [37]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/adikochas/capstone_MLops.mlflow')
dagshub.init(repo_owner='adikochas', repo_name='capstone_MLops', mlflow=True)


# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

c:\Users\adiko\OneDrive\Desktop\final_mlops_proj\capstone_MLops\venv\Lib\site-packages\rich\live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=a798fe9b-d606-46b3-b1df-dcb87724da88&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=0badaae5fb34a1f6cc09a5ddc55a438ceab29ccca7b7f1142ee2ec240ce1841f




Accessing as adikochas

Initialized MLflow to track repo "adikochas/capstone_MLops"

Repository adikochas/capstone_MLops initialized!

2026/04/19 17:42:11 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/35848f7487f74a839fc3b308a0a8e8a5', creation_time=1776600732443, experiment_id='0', last_update_time=1776600732443, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, trace_location=None, workspace='default'>

In [38]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)

2026-04-19 17:42:11,890 - INFO - Starting MLflow run...
2026-04-19 17:42:13,869 - INFO - Logging preprocessing parameters...
2026-04-19 17:42:14,793 - INFO - Initializing Logistic Regression model...
2026-04-19 17:42:14,795 - INFO - Fitting the model...
2026-04-19 17:42:14,851 - INFO - Model training complete.
2026-04-19 17:42:14,852 - INFO - Logging model parameters...
2026-04-19 17:42:15,208 - INFO - Making predictions...
2026-04-19 17:42:15,210 - INFO - Calculating evaluation metrics...
2026-04-19 17:42:15,231 - INFO - Logging evaluation metrics...
2026-04-19 17:42:16,546 - INFO - Saving and logging the model...
2026/04/19 17:42:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/19 17:42:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The rec

🏃 View run unleashed-duck-983 at: https://dagshub.com/adikochas/capstone_MLops.mlflow/#/experiments/0/runs/b786143b00074bb889cb08df6bc79485
🧪 View experiment at: https://dagshub.com/adikochas/capstone_MLops.mlflow/#/experiments/0
